# Week 6 · Class 2 — Linear regression and regularisation, from start to finish 🐟 (live coding)

**2 hours** · we code together · one dataset for the whole session: `Fish.csv`

**The project:** a fish market wants to estimate the **weight** of a fish from measurements that are quick to take (length, height, width) and its species. Weight is a number, so this is a **regression** problem.

| Part | Time | What we do | Why |
|---|---|---|---|
| 0 | 3' | Setup and load the data | |
| 1 | 6' | First look and cleaning | know the data before modelling |
| 2 | 7' | EDA: one variable at a time | shape, skew, outliers |
| 3 | 4' | EDA: the species | does the category matter? |
| 4 | 10' | **Confidence intervals** | how sure are we about an average? |
| 5 | 6' | EDA: relationships | which features move with the weight? |
| 6 | 10' | **Model, iteration 1** (step by step) | a first baseline |
| 7 | 12' | **Log transform** | fix what the baseline gets wrong |
| 8 | 4' | The statistics of the model | confidence intervals for the coefficients |
| 9 | 8' | **Model, iteration 2: the `Pipeline`** | same model, clean and safe code |
| | | ☕ *first half done: 70 minutes* | |
| 10 | 7' | Correlated features: unstable coefficients | the problem regularisation solves |
| 11 | 10' | Polynomial features: more power… and **overfitting** | another way to fit a curve |
| 12 | 10' | **Ridge and Lasso** | keep the power, remove the danger |
| 13 | 10' | The effect of alpha and `GridSearchCV` | choosing the strength of the penalty |
| 14 | 3' | Recap | |

Total: about 110 minutes. If time is short, parts 8 and the two alpha plots of part 13 can be read at home.

🧑‍💻 = we write the code together. 📖 = how to read the result.

---
## 0 · Setup and load the data (3')

**In Google Colab:** click the folder icon on the left and drag `Fish.csv` into it. Or run the next cell without the `#` signs and choose the file.

In [ ]:
## Colab only: remove the two # below, run the cell and choose Fish.csv
# from google.colab import files
# files.upload()

In [ ]:
## import dependencies

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [ ]:
df = pd.read_csv("Fish.csv")
df

**The columns**
- `Species`: the kind of fish (text)
- `Weight`: weight in grams → **our target**
- `Length1`, `Length2`, `Length3`: three ways of measuring the length, in cm
- `Height`, `Width`: in cm

---
## 1 · First look and cleaning (6')

Three questions before anything else: **how big** is the data, **what type** is each column, and **is anything missing or strange**?

🧑‍💻 Size, types and missing values.

In [ ]:
## how many rows and columns?

In [ ]:
## types of the columns

In [ ]:
## missing values per column

In [ ]:
## how many duplicated rows?

🧑‍💻 Descriptive statistics. Look at the **min** of each column.

In [ ]:
## descriptive statistics

📖 The minimum `Weight` is **0 grams**. A fish cannot weigh nothing: this is a **data error**, not a real fish.

A missing value is not always written as `NaN`. Sometimes it hides as an impossible number, and only `describe()` shows it.

In [ ]:
## show the rows where Weight <= 0

In [ ]:
## keep only the rows with Weight > 0 and check the shape

📖 We removed 1 row out of 159. We could not repair it (we do not know the real weight), and one row is a small price.

---
## 2 · EDA: one variable at a time (7')

We start with the **target**. Its shape tells us what kind of model problems to expect.

🧑‍💻 Histogram of the weight.

In [ ]:
## histogram of Weight with kde=True

In [ ]:
## skewness of Weight

📖 The weight is **right-skewed**: many light fish, a few very heavy ones (skewness around 1.1).

Why we care: a straight-line model is pulled by the few heavy fish. Keep this in mind, we come back to it in part 7.

🧑‍💻 Now all the numeric columns at once.

In [ ]:
## list of numeric columns, then df[numeric_columns].hist(...)

🧑‍💻 Outliers: a boxplot of the weight.

In [ ]:
## boxplot of Weight

In [ ]:
## compute Q1, Q3, IQR, upper_limit and show the fish above the limit

📖 The three "outliers" are all **Pike**, a species that really grows that big. They are **real fish**, not errors.

**Rule:** remove a value when it is *wrong* (the 0 g fish). Keep a value when it is *rare but real* (the big pikes). If we deleted them, our model would never learn to weigh a big pike.

---
## 3 · EDA: the species (4')

🧑‍💻 How many fish of each species?

In [ ]:
## count the fish per species

🧑‍💻 Does the weight change with the species?

In [ ]:
## boxplot: Species on x, Weight on y

📖 Yes: Smelt are tiny, Pike and Bream are heavy. The species carries information, so it will be a **feature** of our model.

Also notice: some species have very few fish (Whitefish: 6). Remember this for the next part.

---
## 4 · Confidence intervals (10')

### Why do we need them?
We measured **35 Bream**. The market wants to know the average weight of **all** the Bream in the lake. Our 35 fish are only a **sample**: another catch would give a slightly different average.

A **confidence interval** answers: *"given my sample, in which range is the true average likely to be?"*

It is built from three ingredients:

$$\text{mean} \;\pm\; t \times \frac{\text{std}}{\sqrt{n}}$$

- **mean**: our best guess
- **std / √n**: the **standard error**, how much the mean would move from one sample to another. More fish (bigger n) → smaller error.
- **t**: how many standard errors we take to be 95% confident (about 2)

🧑‍💻 Step by step, for the Bream.

In [ ]:
## select the Weight of the Bream, then compute n, mean and std

In [ ]:
## standard_error = std / square root of n

In [ ]:
## t_value with stats.t.ppf(0.975, df=n - 1)

In [ ]:
## lower and upper bound of the interval

📖 **How to say it:** "We are 95% confident that the average weight of all Bream is between about 546 g and 690 g."

It is a statement about the **average**, not about one fish. A single Bream can weigh 250 g or 1000 g.

🧑‍💻 The same result in one line with `scipy`.

In [ ]:
## stats.t.interval(0.95, df=..., loc=..., scale=...)

🧑‍💻 Now for every species, with a loop.

In [ ]:
## loop over the species: print n, the mean and the 95% interval (stats.sem gives the standard error)

🧑‍💻 `seaborn` draws these intervals for us: the black lines on a barplot **are** 95% confidence intervals.

In [ ]:
## barplot of Weight per Species with errorbar=('ci', 95)

📖 **Reading the intervals**
- **Whitefish** has a huge interval (about 200 g to 850 g): only 6 fish, so we know very little about its true average.
- **Smelt** has a tiny interval: the fish are all alike, so 14 fish are enough.
- **Bream vs Roach**: the intervals do not overlap → the averages are really different.
- **Bream vs Pike**: the intervals overlap → with this data we cannot say which species is heavier on average.

**Why this matters for modelling:** an average without its interval hides how much data is behind it. In part 8 we use the same idea for the **coefficients** of the model.

---
## 5 · EDA: relationships between the variables (6')

🧑‍💻 Weight against length, one colour per species.

In [ ]:
## scatterplot: Length1 on x, Weight on y, hue = Species

📖 Two things to see:
1. The longer the fish, the heavier: a strong relationship.
2. The cloud is **curved**, like a banana. A straight line will not follow it well. Remember this for part 7.

🧑‍💻 Correlations between all the numeric columns.

In [ ]:
## heatmap of the correlation matrix with annot=True

📖 `Length1`, `Length2` and `Length3` have correlations above **0.99** with each other. They tell the same story three times.

This is called **multicollinearity**. For a linear model it is a problem: the model cannot decide which of the three deserves the credit, and the coefficients become unstable.

**Decision:** we keep `Length1` and drop the other two.

In [ ]:
## drop Length2 and Length3

---
## 6 · Model, iteration 1: step by step (10')

The recipe from Tuesday:

1. choose the target `y` and the features `X`
2. turn the text column into numbers (**dummies**)
3. **split** into train and test
4. **standardise** the features
5. **fit** the model on the train set
6. **evaluate** on train and on test

🧑‍💻 Steps 1 and 2.

In [ ]:
## X = all the columns except Weight, y = Weight

In [ ]:
## dummies for the text column

🧑‍💻 Step 3: the split. The test set stays hidden until the evaluation.

In [ ]:
## train_test_split with test_size=0.2 and random_state=42

🧑‍💻 Step 4: standardise. For each column: `(value - mean) / std`.

⚠️ The mean and std are learned on the **train** set only (`fit_transform`), then applied to the test set (`transform`). The test set must not influence anything.

In [ ]:
## StandardScaler: fit_transform on X_train, transform on X_test

🧑‍💻 Steps 5 and 6: fit and evaluate.

In [ ]:
## create a LinearRegression and fit it on the scaled train set

In [ ]:
## predictions on train and test, then r2 on both and the mean absolute error on test

📖 R² is about 0.93 on train and 0.94 on test. Train ≈ test, so **no overfitting**. On average the model is wrong by about 65 g.

It looks good. But a score never tells the whole story: let's look at the errors.

🧑‍💻 Residuals = real weight − predicted weight. For a good model they form a cloud around 0 with **no shape**.

In [ ]:
## residuals_1 = y_test - predictions_test, then scatter plot against the predictions with a red line at 0

In [ ]:
## smallest prediction, and how many predictions are negative

📖 Two problems hidden behind the good R²:
1. The residuals make a **U shape**: positive for small and big fish, negative in the middle. The model draws a straight line through a curve.
2. Some predictions are **negative**. The model says a small fish weighs less than 0 grams!

Both come from the same cause: the relationship between size and weight is **not a straight line**.

---
## 7 · The log transform (12')

### Why would a logarithm help?
Think about the shape of a fish. Its weight depends on its **volume**, and a volume is

$$\text{weight} \approx c \times \text{length} \times \text{height} \times \text{width}$$

This is a **multiplication**, and a linear model can only **add**. The logarithm turns one into the other:

$$\log(\text{weight}) \approx \log(c) + \log(\text{length}) + \log(\text{height}) + \log(\text{width})$$

After taking logs, the relationship is a sum: exactly what a linear regression can learn.

🧑‍💻 See it: the same scatter plot, before and after the log.

In [ ]:
## two plots side by side: Weight vs Length1, and log(Weight) vs log(Length1)

📖 The banana became a straight line. The log did not change the data, it changed the **ruler** we use to look at it.

🧑‍💻 Now the model. Same recipe as iteration 1, with logs on the three size columns and on the target.

In [ ]:
## copy X into X_log, take np.log of Length1, Height and Width; y_log = np.log(y)

In [ ]:
## train_test_split on X_log and y_log with the same test_size and random_state

In [ ]:
## scale (fit on train only), then fit a new LinearRegression called model_2

🧑‍💻 The model now predicts **log(weight)**. Nobody buys fish in "log grams", so we go back to grams with the opposite of the log: `np.exp`.

In [ ]:
## predict on the test set, then np.exp to go back to grams

🧑‍💻 Compare both models **in grams**, on the same test fish.

In [ ]:
## print r2, mean absolute error and number of negative predictions for both models (use y_test in grams for both)

📖 **The benefit of the log transform, in numbers**
- R² on the test set goes from about **0.94 to 0.98**.
- The average error is **cut in half**: from about 65 g to about 30 g.
- **No negative prediction** any more: `np.exp` of any number is always positive.

Same data, same algorithm. We only changed how we present the data to the model.

🧑‍💻 And the residuals?

In [ ]:
## residuals of the log model, then the two residual plots side by side

📖 The U shape is gone: the residuals are now a cloud around 0. This is what a healthy linear model looks like.

**When to think of a log transform**
- the target or a feature is **right-skewed** (money, weights, counts, populations)
- the scatter plot is **curved** and opens like a fan
- the quantities **multiply** each other
- the values must be **strictly positive** (log of 0 does not exist: one more reason we removed the 0 g fish)

---
## 8 · The statistics of the model (4')

`scikit-learn` gives us the coefficients. `statsmodels` gives us the coefficients **with their confidence intervals**: the same idea as part 4, applied to the model.

To read them easily we use only the three size columns, in logs, without scaling.

In [ ]:
## statsmodels: add_constant on the three size columns, sm.OLS(...).fit(), then print the summary

In [ ]:
## confidence intervals of the coefficients

📖 **How to read the table**
- `coef`: the coefficient. In a log–log model it reads in **percent**: +1% of length goes with about +1.4% of weight.
- `[0.025  0.975]`: the 95% confidence interval of the coefficient. If it does **not contain 0**, the feature has a real effect.
- `P>|t|`: the p-value. Below 0.05 means the same thing: the effect is unlikely to be chance.

**A nice check:** add the three coefficients. You get about **3**. Double every dimension of a fish and its weight is multiplied by 2³ = 8: the model has found the formula of a **volume** by itself.

In [ ]:
## sum of the three coefficients

---
## 9 · Model, iteration 2: the `Pipeline` (8')

Look back at what we did by hand: log, dummies, split, scaler on train, scaler on test, fit, predict. Many steps, many variables (`X_train_log_scaled`...), many chances to make a mistake, like scaling the test set with its own mean.

A **`Pipeline`** puts all the steps in **one object**:
- `pipeline.fit(X_train, y_train)` → every step learns from the **train** set only
- `pipeline.predict(X_test)` → the same steps are applied, in the same order

Same model, same results. Cleaner code, and no data leakage by construction.

🧑‍💻 We start again from the raw columns.

In [ ]:
## two lists of columns, X with the raw columns, y_log, then the same split

🧑‍💻 The preprocessing. Each kind of column gets its own treatment:
- numeric columns: **log**, then **scale**
- text column: **dummies** (in a pipeline this is called `OneHotEncoder`)

`ColumnTransformer` sends each column to the right treatment.

In [ ]:
## numeric_steps = log then scaler; categorical_steps = OneHotEncoder; ColumnTransformer with both

🧑‍💻 The full pipeline: preprocessing, then the model.

In [ ]:
## pipeline = make_pipeline(preprocessing, LinearRegression())

In [ ]:
## fit the pipeline, predict (np.exp to go back to grams), print r2 and mean absolute error in grams

📖 Exactly the same score as in part 7: it is the **same model**, written in a cleaner way.

Now the real benefit: predicting a **new fish** from its raw measurements takes one line.

In [ ]:
## a DataFrame with one new fish (30, 12, 5, Bream), then predict its weight in grams

🧑‍💻 Bonus: **cross-validation**. The data is split 5 times, and each time the whole pipeline is trained again on the train part only. This is only safe because the scaler lives **inside** the pipeline.

Our file is sorted by species, so we ask for the folds to be **shuffled** first.

In [ ]:
## KFold with 5 shuffled folds, cross_val_score(pipeline, X, y_log, cv=folds), print the scores and their mean

---
## 10 · Correlated features: unstable coefficients (7')

☕ **Second half.** So far our best model is the log model: test R² 0.98 with a handful of features.

In part 5 we **dropped** `Length2` and `Length3` because they repeat `Length1`. What would have happened if we had kept them? This is the problem that **regularisation** was invented to solve.

🧑‍💻 We start again from the file, with **all** the columns, in grams (no log).

In [ ]:
## read Fish.csv again, remove the 0 g fish, X = dummies of all the columns except Weight, y = Weight, same split

🧑‍💻 We are going to train many models. To avoid repeating the same lines, we write **one function** that does the three steps: transform the features, fit the model, print the scores.

In [ ]:
## def model_inplace(transformer, model, X_train, X_test, y_train, y_test): transform, fit, print r2 on train and test

In [ ]:
## baseline_model = LinearRegression(); scaler1 = StandardScaler(); call model_inplace

📖 The score is fine, about the same as iteration 1. Now look at the **coefficients** of the three lengths.

In [ ]:
## a DataFrame with the coefficients of the model, indexed by X.columns

📖 `Length1` gets about **−630** and `Length2` about **+700**. The model says: "a longer fish is much lighter… and much heavier". The two almost cancel out.

The three lengths carry the same information, so the model cannot decide how to share the credit. It finds huge coefficients with opposite signs that happen to fit the training fish. This is **multicollinearity** in action:
- the **predictions** are still fine
- the **coefficients** mean nothing, and they would change completely with a few different fish

Dropping columns by hand (part 5) is one solution. Regularisation is the automatic one. First, let's make the problem worse.

---
## 11 · Polynomial features: more power… and overfitting (10')

In part 7 we fixed the curve with a **log**, because we understood the data (a weight is a volume).

There is another way, which needs no understanding: give the model new columns made of **squares and products** of the existing ones (`Length1²`, `Length1 × Height`, …) and let it build the curve itself. This is what `PolynomialFeatures` does.

We chain it with the scaler in a small pipeline: scale → create the polynomial columns → scale again.

In [ ]:
## poly2 and poly3: make_pipeline(StandardScaler(), PolynomialFeatures(degree=...), StandardScaler())

In [ ]:
## number of training fish, and number of columns: original, degree 2, degree 3

📖 From 12 columns to 90, then to 454. And we have 126 fish to learn from: with degree 3, **more features than fish**.

In [ ]:
## baseline model with poly2

In [ ]:
## baseline model with poly3

📖 **Degree 2:** train and test both go up. The model built the curve by itself, without any log.

**Degree 3:** the train score is almost a perfect 1… and the test score **collapses** to a negative number. A negative R² means the model is worse than answering "the average weight" for every fish.

With 454 features for 126 fish, the model has enough freedom to learn every training fish **by heart**, noise included. This is **overfitting**.

🧑‍💻 What does a collapsed model predict? Real fish weigh between 6 g and 1,650 g.

In [ ]:
## fit the baseline model on the poly3 features and print the smallest and biggest prediction on the test set

📖 A fish of minus several kilos. On the training fish the model is perfect; on a fish it has never seen, it can say anything.

🧑‍💻 The picture to remember: train against test for the three models.

In [ ]:
## loop over the three transformers: fit a LinearRegression, store the r2 on train and on test in a DataFrame called scores

In [ ]:
## bar chart of the scores DataFrame (use scores.clip(lower=0) so a negative score is drawn as 0)

📖 **The gap between the two bars is the signature of overfitting.** The train bar alone would have told us degree 3 is the best model. It is the worst.

---
## 12 · Ridge and Lasso (10')

Both problems we just saw, the lengths with −630 and +700 and the collapsed degree 3, have the same symptom: **huge coefficients**.

Regularisation adds a **penalty** for big coefficients to what the model minimises:

- **Ridge (L2):** errors + `alpha` × (sum of the **squared** coefficients)
- **Lasso (L1):** errors + `alpha` × (sum of the **absolute** coefficients)

`alpha` is the strength of the penalty: `alpha = 0` is a plain linear regression; a very large `alpha` pushes every coefficient towards 0.

⚠️ The penalty looks at the **size** of the coefficients, so the features must be on the same scale: **always scale before regularising**.

In [ ]:
## import Ridge and Lasso; ridge = Ridge(alpha=10); lasso = Lasso(alpha=1, max_iter=200000)

🧑‍💻 The same 454 features of degree 3 that collapsed, now with a penalty.

In [ ]:
## ridge with poly3

In [ ]:
## lasso with poly3

📖 From a negative score back to about **0.98** on the test set. The train score went down a little: that is the price, and it is a good deal.

We kept the flexible model and removed its freedom to do silly things.

In [ ]:
## how many coefficients are different from 0 for ridge and for lasso?

📖 The big difference between the two:
- **Ridge** keeps all 454 features, with small coefficients.
- **Lasso** sets most coefficients to **exactly 0**: it keeps fewer than 30 features out of 454. Lasso does **feature selection** for us.

🧑‍💻 **See it.** Back to the 12 original features and to the three lengths of part 10: what do Ridge and Lasso do with them?

In [ ]:
## scale with scaler1, fit a LinearRegression, a Ridge(alpha=10) and a Lasso(alpha=5); put the three coef_ in a DataFrame indexed by X.columns

In [ ]:
## bar chart of the coefficients DataFrame

📖 Look at the three lengths:
- **Linear:** about −630, +700, +370. Huge, opposite signs, meaningless.
- **Ridge:** about +97, +99, +97. It **shares the credit equally** between the three lengths, which is exactly right: they measure the same thing.
- **Lasso:** it sets `Length1` to **0** and keeps the other two. It does by itself what we did by hand in part 5: drop a redundant column.

Ridge is the good choice when the correlated features all matter a little. Lasso is the good choice when you want a short list of features.

---
## 13 · The effect of alpha and `GridSearchCV` (10')

How do the coefficients change when we turn the `alpha` knob? We fit a Lasso for many values of alpha and store the coefficients each time.

In [ ]:
## alphas = np.logspace(-1, 2.7, 50); loop: fit a Lasso for each alpha, store coef_, the train score and the test score in three lists

In [ ]:
## plot the coefficients against alpha with a log scale on x; one line per feature

📖 Read it from left to right, like turning a knob:
- On the left (small alpha): almost a plain linear regression, with the big opposite coefficients of the lengths.
- Moving right: the lines reach 0 **one after the other**. The features the model needs least disappear first.
- On the far right: everything is 0. The model predicts the average weight for every fish.

In [ ]:
## plot the train and test scores against alpha (log scale on x)

📖 With too much penalty (right side) both scores fall: the model is too simple, it **underfits**. Too little penalty and we are back to overfitting. The best alpha is in between.

How do we find it **without looking at the test set**?

### `GridSearchCV`
`alpha` is a **hyperparameter**: a setting we choose, not something the model learns.

`GridSearchCV` is a `for` loop over the values we give it. For each one it does a **cross-validation** inside the train set (here 5 folds) and keeps the value with the best average score. The test set is never used.

We search on the 454 features of degree 3, where the penalty really matters.

In [ ]:
## poly3 features, params = {'alpha': [...]}, GridSearchCV(Lasso(max_iter=200000), param_grid=params, cv=5), fit on train, print best_params_

In [ ]:
## best_model = grid.best_estimator_; print its scores on train and test, and how many features it kept

📖 `GridSearchCV` chose the alpha by itself, using only the training fish, and the model it returns scores about 0.97 on the test set with only a few of the 454 features.

**Now compare the two roads we took today:**

| Road | Features | Test R² |
|---|---|---|
| Understanding the data: **log transform** (part 7) | 10 | **0.98** |
| Brute force: degree 3 polynomials, no penalty (part 11) | 454 | collapses |
| Brute force + **Lasso** (this part) | about 10 kept out of 454 | 0.97 |

Regularisation saved the brute-force model. But the simple model built on an understanding of the data (a weight is a volume) is just as good, and far easier to explain. **Regularisation is a safety net, not a replacement for knowing your data.**

---
## 14 · Recap (3')

| Step | What we learned |
|---|---|
| Cleaning | `describe()` reveals impossible values (0 g). Remove what is wrong, keep what is rare but real. |
| EDA | The target is right-skewed, the relationship with length is curved, the three lengths are redundant. |
| Confidence interval | mean ± t × std/√n. Few data → wide interval. It also exists for model coefficients. |
| Iteration 1 | R² 0.94 looked fine, but the residuals had a U shape and some predictions were negative. |
| Log transform | Turns a multiplication into a sum. Test R² 0.94 → 0.98, error cut in half, no negative prediction. |
| Pipeline | All the steps in one object: less code, no leakage, new predictions and cross-validation in one line. |
| Multicollinearity | Redundant features → huge coefficients with opposite signs. Predictions fine, coefficients meaningless. |
| Overfitting | Many features, few rows: train score up, test score down. Look at the **gap**. |
| Ridge (L2) | Shrinks all the coefficients and shares the credit between correlated features. |
| Lasso (L1) | Sets some coefficients to exactly 0: automatic **feature selection**. |
| alpha | The strength of the penalty. Too small: overfitting. Too big: underfitting. Chosen with `GridSearchCV`. |

**The habit to keep:** model → look at the residuals and at the train/test gap → understand what is wrong → improve → repeat.

➡️ **Homework:** you apply all of this, step by step, to a new dataset.